In [1]:
from spot import SpotEnv
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from typing import Callable, Optional

def make_env(
    render_mode: Optional[str] = None,
    port: int = 0,
    sync_real_time: bool = False,
) -> Callable:
    def _init() -> gym.Env:
        env = SpotEnv(render_mode, port, sync_real_time)
        env = TimeLimit(env, max_episode_steps=2000)
        return env

    return _init

In [2]:
num_envs = 8
train_env = SubprocVecEnv([make_env() for _ in range(num_envs)])
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)

In [3]:
from typing import Callable

def linear_schedule_3(start: float, end: float, duration: float = 1.0) -> Callable[[float], float]:
    """
    Linearly interpolate from `start` to `end` over the first `duration` fraction of training.
    `progress_remaining` goes from 1.0 (start) to 0.0 (end) in SB3.
    """
    assert duration > 0.0, "duration must be > 0"
    def func(progress_remaining: float) -> float:
        # Convert to elapsed fraction t in [0, 1]
        t = 1.0 - progress_remaining
        x = min(max(t / duration, 0.0), 1.0)
        return start + x * (end - start)
    return func


# model = PPO(
#     "MlpPolicy",
#     train_env,
#     device="cpu",
#     tensorboard_log="./tensorboard",
#     verbose=1,
# )

model = SAC(
    "MlpPolicy",
    train_env,
    device="cuda",
    tensorboard_log="./tensorboard",
    verbose=1,
)

Using cuda device


In [4]:
model = PPO.load("./models/spot_ppo_model", env=train_env)

/home/kane/Documents/code/other/personal_website/.venv/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [4]:
evaluation_env = DummyVecEnv([make_env(render_mode="human") for _ in range(1)])
evaluation_env = VecNormalize(evaluation_env, norm_obs=True, norm_reward=True, training=False, gamma=train_env.gamma, clip_obs=10.0, clip_reward=10.0)

# Callback to stop training if no improvement
stop_callback = StopTrainingOnNoModelImprovement(max_no_improvement_evals=5, min_evals=5, verbose=1)

eval_callback = EvalCallback(
    evaluation_env,
    callback_after_eval=stop_callback,
    eval_freq=5000,  # Evaluate every 5000 steps
    n_eval_episodes=5,
    best_model_save_path="./models/",
    log_path="./logs/",
    deterministic=True,
    render=False,
    verbose=1
)

[WebUI] Listening at http://localhost:36377


In [ ]:
# takes around 100 minutes
model.learn(total_timesteps=500000, callback=eval_callback)

Logging to ./tensorboard/SAC_0


[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the e

/home/kane/Documents/code/other/personal_website/.venv/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

Eval num_timesteps=80000, episode_reward=-8.94 +/- 2.99
Episode length: 1418.20 +/- 488.35
---------------------------------
| eval/              |          |
|    mean_ep_length  | 1.42e+03 |
|    mean_reward 

: 

In [ ]:
model.save("models/spot_ppo_model")

In [ ]:
# save the normalization statistics
train_env.save("./models/vec_normalize.pkl")
train_env.close()

In [5]:
test_env = DummyVecEnv([make_env(render_mode="human", port=24546, sync_real_time=True)])
test_env = VecNormalize.load("./models/vec_normalize.pkl", test_env)
test_env.training = False
test_env.norm_reward = False

[WebUI] Listening at http://localhost:24546


In [8]:
terminated = False
obs = test_env.reset()
while not terminated:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, info = test_env.step(action)

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size

[warning] Rescheduling timed events. These are the events that the StatePropgator will try to reschedule:
Before hop events:

After hop events:
    hop_size



In [15]:
test_env.close()

[warning] Object 'frame_container (Karana::Frame::FrameContainer)' with id=1 is not destroyed and is not a zombie.

[warning] Object 'root (Karana::Frame::Frame)' with id=2 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=4 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=6 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=8 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=10 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=12 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=14 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=16 is not destroyed and is not a zombie.

[warning] Object 'static_mesh (Karana::Core::Base)' with id=18 is not destroy

AssertionError: 